In [1]:
import os

# Limit CPU threads for major C/C++ underlying libraries
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import torch
torch.set_num_threads(1)  # Specifically limits PyTorch CPU threads

### Test Reward Model

In [2]:

import torch
import random
import regex as re
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

def set_seed(seed: int = 42):
    """Locks all random seeds for complete reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If you are using a GPU, you also need to lock the CUDA seeds
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces cuDNN to use deterministic algorithms
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Call this before doing anything else!
set_seed(42)

/home/ttsi/miniconda3/envs/spec_sampl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.resolve()))

In [4]:
from reward_functions import reward_function
from data.data_helper import *
from hugging_face_models_functions import *

In [5]:
data = Data(split='train')
df = data.df
df.head()

,question,answer
0,Natalia sold clips to 48 of her friends in Apr...,Natalia sold 48/2 = <<48/2=24>>24 clips in May...
1,Weng earns $12 an hour for babysitting. Yester...,Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...
2,Betty is saving money for a new wallet which c...,"In the beginning, Betty has only 100 / 2 = $<<..."
3,"Julie is reading a 120-page book. Yesterday, s...",Maila read 12 x 2 = <<12*2=24>>24 pages today....
4,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...


In [6]:
model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"
system_instruction = "Just give me one number as the answer, no explanation, no text, no words, just the number."

response = hugging_face_models_functions(model_name=model_name, input_text=df['question'].iloc[0], system_instruction=system_instruction)
print("Response from the model:", response)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1149.05it/s]


<|im_start|>system
Just give me one number as the answer, no explanation, no text, no words, just the number.<|im_end|>
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant

Response from the model: Natalia sold 48 clips in April to 48 friends.
In May, she sold half as many clips as she did in April.
So, Natalia sold 48 / 2 = 24 clips in May to 24 friends.
Therefore, Natalia sold a total of 48 + 24 = 72 clips in April and May.
#### 72
The answer is: 72


In [ ]:
df['answer'].iloc[0]

'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'